In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_PATH = Path("..")

PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
RAW_PATH = PROJECT_PATH / "data" / "raw"

print("Project path:")
print(PROJECT_PATH.resolve())

print("\nRaw data path:")
print(RAW_PATH.resolve())

print("\nProcessed data path:")
print(PROCESSED_PATH.resolve())

Project path:
D:\fraud-loss-analysis-step1\fraud-loss-analysis

Raw data path:
D:\fraud-loss-analysis-step1\fraud-loss-analysis\data\raw

Processed data path:
D:\fraud-loss-analysis-step1\fraud-loss-analysis\data\processed


In [2]:
# ============================================================
# CELL 2: LOAD VALIDATION DATA + HIDDEN GROUND TRUTH
# ============================================================

transactions = pd.read_csv(
    PROCESSED_PATH / "fact_transaction_clean.csv"
)

transactions["txn_ts"] = pd.to_datetime(
    transactions["txn_ts"],
    errors="coerce",
    format="mixed"
)

ground_truth = pd.read_csv(
    RAW_PATH / "sim_ground_truth.csv"
)

print("Transactions shape:", transactions.shape)
print("Ground truth shape:", ground_truth.shape)

print("\nTransaction columns:")
print(transactions.columns.tolist())

print("\nGround truth columns:")
print(ground_truth.columns.tolist())

print("\nGround truth typology counts:")
print(ground_truth["typology"].value_counts())

Transactions shape: (1498353, 14)
Ground truth shape: (10500, 2)

Transaction columns:
['txn_id', 'card_id', 'customer_id', 'merchant_id', 'device_id', 'txn_ts', 'amount', 'channel', 'category', 'merchant_city', 'merchant_country', 'ip_country', 'auth_result', 'customer_home_city']

Ground truth columns:
['txn_id', 'typology']

Ground truth typology counts:
typology
account_takeover          2135
card_testing              2034
cnp_high_risk_merchant    1826
compromised_terminal      1527
cross_border_spike        1492
impossible_travel         1486
Name: count, dtype: int64


In [3]:
# ============================================================
# CELL 3: CREATE VALIDATION DATASET
# ============================================================

validation_transactions = transactions[
    transactions["txn_ts"].dt.month.isin([9, 10, 11, 12])
].copy()

print("Validation transactions:", len(validation_transactions))

print(
    "Validation period:",
    validation_transactions["txn_ts"].min(),
    "to",
    validation_transactions["txn_ts"].max()
)

print(
    "\nValidation transaction value:",
    f"₹{validation_transactions['amount'].sum():,.2f}"
)

Validation transactions: 500414
Validation period: 2025-09-01 00:00:47 to 2025-12-31 23:59:35

Validation transaction value: ₹1,333,617,606.22


In [4]:
# ============================================================
# CELL 4: MERGE HIDDEN FRAUD TYPOLOGY
# ============================================================

validation_with_truth = validation_transactions.merge(
    ground_truth,
    on="txn_id",
    how="left"
)

print("Rows after merge:", len(validation_with_truth))

print("\nTypology missing values:")
print(validation_with_truth["typology"].isna().sum())

print("\nValidation typology distribution:")
print(
    validation_with_truth["typology"]
    .value_counts(dropna=False)
)

Rows after merge: 500414

Typology missing values:
496890

Validation typology distribution:
typology
NaN                       496890
card_testing                 716
account_takeover             693
cnp_high_risk_merchant       622
cross_border_spike           509
compromised_terminal         506
impossible_travel            478
Name: count, dtype: int64


In [5]:
# ============================================================
# CELL 5: APPLY SELECTED FRAUD RULE
# ============================================================

SELECTED_THRESHOLD = 20000

validation_with_truth["rule_alert"] = (
    (validation_with_truth["channel"] == "E_COMMERCE")
    & (
        validation_with_truth["category"].isin(
            ["Electronics", "Gift Cards"]
        )
    )
    & (
        validation_with_truth["amount"] >= SELECTED_THRESHOLD
    )
).astype(int)

print("Selected rule:")
print("E_COMMERCE + Electronics/Gift Cards + Amount >= ₹20,000")

print("\nTotal validation transactions:")
print(f"{len(validation_with_truth):,}")

print("\nRule alerts:")
print(
    f"{validation_with_truth['rule_alert'].sum():,}"
)

print("\nAlert rate:")
print(
    f"{100 * validation_with_truth['rule_alert'].mean():.2f}%"
)

Selected rule:
E_COMMERCE + Electronics/Gift Cards + Amount >= ₹20,000

Total validation transactions:
500,414

Rule alerts:
616

Alert rate:
0.12%


In [6]:
# ============================================================
# CELL 6: HIDDEN GROUND-TRUTH CONFUSION MATRIX
# ============================================================

validation_with_truth["hidden_fraud"] = (
    validation_with_truth["typology"].notna()
).astype(int)

tp = (
    (validation_with_truth["rule_alert"] == 1)
    & (validation_with_truth["hidden_fraud"] == 1)
).sum()

fp = (
    (validation_with_truth["rule_alert"] == 1)
    & (validation_with_truth["hidden_fraud"] == 0)
).sum()

fn = (
    (validation_with_truth["rule_alert"] == 0)
    & (validation_with_truth["hidden_fraud"] == 1)
).sum()

tn = (
    (validation_with_truth["rule_alert"] == 0)
    & (validation_with_truth["hidden_fraud"] == 0)
).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)

print("Hidden Ground-Truth Evaluation")
print("=" * 40)

print(f"True Positives  : {tp:,}")
print(f"False Positives : {fp:,}")
print(f"False Negatives : {fn:,}")
print(f"True Negatives  : {tn:,}")

print("\nMetrics")
print("=" * 40)

print(f"Precision       : {precision * 100:.2f}%")
print(f"Recall          : {recall * 100:.2f}%")
print(f"False Positive Rate: {fpr * 100:.2f}%")
print(f"F1 Score        : {f1 * 100:.2f}%")

Hidden Ground-Truth Evaluation
True Positives  : 432
False Positives : 184
False Negatives : 3,092
True Negatives  : 496,706

Metrics
Precision       : 70.13%
Recall          : 12.26%
False Positive Rate: 0.04%
F1 Score        : 20.87%


In [7]:
# ============================================================
# CELL 7: TYPOLOGY-LEVEL RULE PERFORMANCE
# ============================================================

typology_performance = (
    validation_with_truth[
        validation_with_truth["hidden_fraud"] == 1
    ]
    .groupby("typology")
    .agg(
        fraud_transactions=("txn_id", "count"),
        alerts_caught=("rule_alert", "sum"),
        total_fraud_value=("amount", "sum"),
        fraud_value_caught=(
            "amount",
            lambda x: x[
                validation_with_truth.loc[x.index, "rule_alert"] == 1
            ].sum()
        )
    )
    .reset_index()
)

typology_performance["recall_pct"] = (
    100
    * typology_performance["alerts_caught"]
    / typology_performance["fraud_transactions"]
)

typology_performance["fraud_value_coverage_pct"] = (
    100
    * typology_performance["fraud_value_caught"]
    / typology_performance["total_fraud_value"]
)

typology_performance = typology_performance.sort_values(
    "total_fraud_value",
    ascending=False
)

display(typology_performance)

,typology,fraud_transactions,alerts_caught,total_fraud_value,fraud_value_caught,recall_pct,fraud_value_coverage_pct
0,account_takeover,693,429,21956097.34,18457704.55,61.904762,84.066418
5,impossible_travel,478,1,3869297.35,29242.55,0.209205,0.755759
3,compromised_terminal,506,1,2860627.61,30320.07,0.197628,1.059910
2,cnp_high_risk_merchant,622,0,1698068.01,0.00,0.000000,0.000000
4,cross_border_spike,509,1,1386601.00,32934.81,0.196464,2.375219
1,card_testing,716,0,18493.85,0.00,0.000000,0.000000


In [8]:
# ============================================================
# CELL 8: SAVE SELECTED RULE VALIDATION RESULTS
# ============================================================

selected_rule_results = validation_with_truth[
    validation_with_truth["rule_alert"] == 1
].copy()

selected_rule_results = selected_rule_results[
    [
        "txn_id",
        "card_id",
        "customer_id",
        "merchant_id",
        "txn_ts",
        "amount",
        "channel",
        "category",
        "merchant_country",
        "ip_country",
        "typology",
        "rule_alert"
    ]
]

output_path = (
    PROCESSED_PATH /
    "selected_rule_validation.csv"
)

selected_rule_results.to_csv(
    output_path,
    index=False
)

print("Selected-rule validation saved:")
print(output_path.resolve())

print("\nRows saved:", len(selected_rule_results))

Selected-rule validation saved:
D:\fraud-loss-analysis-step1\fraud-loss-analysis\data\processed\selected_rule_validation.csv

Rows saved: 616
